In [1]:
# ============================================================
# 19_AURORA_probability_skill_observability_bootstrap_sensitivity.ipynb
# Corrected single-cell standalone version
#
# Purpose:
# 1. Generate Table S25b: probability-skill baselines
# 2. Generate Table S41: market-state variable observability and lag convention
# 3. Generate Table S42: bootstrap block-length sensitivity
# 4. Generate Table S42b: compact bootstrap block-length summary
#
# Fixes:
# - Robust date parsing for modeling label files.
# - Robust search for source-aware return matrix.
# - Does not crash if the main source-aware return matrix is missing;
#   it still generates available diagnostics and writes warning files.
#
# Outputs:
# - table_S25b_probability_skill_baselines.csv
# - table_S25b_probability_skill_baselines_rounded.csv
# - table_S41_feature_observability_lag_convention.csv
# - table_S42_bootstrap_block_length_sensitivity.csv
# - table_S42_bootstrap_block_length_sensitivity_rounded.csv
# - table_S42b_bootstrap_block_length_summary.csv
# - table_S42b_bootstrap_block_length_summary_rounded.csv
# - NOTEBOOK19_validation_report.json
# - NOTEBOOK19_file_manifest_SHA256.csv
#
# Research backtest only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import re
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

try:
    from sklearn.metrics import f1_score, balanced_accuracy_score
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False
    print("sklearn not available. Fallback metric implementations will be used.")

# ============================================================
# 1. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "probability_skill_observability_bootstrap_sensitivity" / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAG_DIR = RUN_ROOT / "diagnostics"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAG_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = [0, 1, 2, 3, 4]
K_CLASSES = len(CLASS_LABELS)

STRICT_START = "2024-11-27"
STRICT_END = "2026-03-25"

ANNUALIZATION_DAYS = 252

BOOTSTRAP_REPLICATIONS = 5000
BOOTSTRAP_BLOCK_LENGTHS = [5, 10, 20, 40, 60]
BOOTSTRAP_RANDOM_SEED = 20260718

RANDOM_BASELINE_REPLICATIONS = 1000
RANDOM_BASELINE_SEED = 20260718

# Optional manual overrides.
# If the source-aware return matrix is not found automatically, paste the exact path here.
MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH = None
# Example:
# MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH = "/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/.../notebook13B_source_aware_strict_test_return_matrix.parquet"

# If Notebook 18 return file is not found automatically, paste the exact path here.
MANUAL_NOTEBOOK18_RETURNS_PATH = None
# Example:
# MANUAL_NOTEBOOK18_RETURNS_PATH = "/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_exposure_matched_lambda_reduced_universe/run_20260717_082548/returns/all_notebook18_returns.parquet"

print("=" * 100)
print("AURORA-TWETF Notebook 19")
print("Probability skill baselines, feature observability, and bootstrap block-length sensitivity")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("NOTEBOOK08_INPUT_INDEX:", NOTEBOOK08_INPUT_INDEX)
print("=" * 100)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Missing probability input index: {NOTEBOOK08_INPUT_INDEX}")

# ============================================================
# 2. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def normalize_name(x):
    return re.sub(r"[^A-Za-z0-9]+", "", str(x)).lower()

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("=", "_")
    )

def write_table(df, filename_stem):
    local_csv = TABLE_RUN_DIR / f"{filename_stem}.csv"
    global_csv = TABLE_DIR / f"{filename_stem}_{RUN_ID}.csv"
    df.to_csv(local_csv, index=False)
    df.to_csv(global_csv, index=False)
    print("Saved:", local_csv)
    print("Saved:", global_csv)
    return local_csv, global_csv

def write_rounded_table(df, filename_stem, digits=6):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    return write_table(out, f"{filename_stem}_rounded")

def looks_like_date_series(s):
    parsed = pd.to_datetime(s, errors="coerce")
    if not isinstance(parsed, pd.Series):
        parsed = pd.Series(parsed)
    if parsed.notna().mean() < 0.50:
        return False, parsed
    years = parsed.dt.year
    valid_year = years.between(1990, 2035)
    if valid_year.mean() < 0.50:
        return False, parsed
    if parsed.nunique(dropna=True) < min(10, max(2, len(parsed) // 10)):
        return False, parsed
    return True, parsed

def set_datetime_index_flex(df):
    """
    Robustly set a financial date index.

    Priority:
    1. Existing DatetimeIndex with plausible years.
    2. Columns named date/datetime/timestamp/time/Unnamed: 0/index.
    3. Any column that parses to plausible financial dates.
    4. Existing index if it parses to plausible financial dates.
    """
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        year_series = pd.Series(df.index.year)
        if year_series.between(1990, 2035).mean() > 0.50:
            df.index = pd.to_datetime(df.index)
            df.index.name = "date"
            df = df[~df.index.isna()]
            return df.sort_index()

    preferred_cols = [
        "date", "Date", "DATE",
        "datetime", "Datetime", "DATETIME",
        "timestamp", "Timestamp", "TIMESTAMP",
        "time", "Time", "TIME",
        "Unnamed: 0", "index", "Index",
    ]
    candidate_cols = [c for c in preferred_cols if c in df.columns] + [
        c for c in df.columns if c not in preferred_cols
    ]

    for c in candidate_cols:
        try:
            ok, parsed = looks_like_date_series(df[c])
            if ok:
                df = df.drop(columns=[c])
                df.index = pd.to_datetime(parsed)
                df.index.name = "date"
                df = df[~df.index.isna()]
                return df.sort_index()
        except Exception:
            continue

    idx_series = pd.Series(df.index)
    ok, parsed_idx = looks_like_date_series(idx_series)
    if ok:
        df.index = pd.to_datetime(parsed_idx.values)
        df.index.name = "date"
        df = df[~df.index.isna()]
        return df.sort_index()

    return df

def read_table_auto_flex(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    df = set_datetime_index_flex(df)
    return df

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "date"})

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

# ============================================================
# 3. Probability-skill metric functions
# ============================================================

def macro_f1_score_safe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    if HAS_SKLEARN:
        return float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                labels=CLASS_LABELS,
                zero_division=0,
            )
        )

    f1s = []
    for k in CLASS_LABELS:
        tp = np.sum((y_true == k) & (y_pred == k))
        fp = np.sum((y_true != k) & (y_pred == k))
        fn = np.sum((y_true == k) & (y_pred != k))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1s.append(f1)

    return float(np.mean(f1s))

def balanced_accuracy_score_safe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    if HAS_SKLEARN:
        return float(balanced_accuracy_score(y_true, y_pred))

    recalls = []
    for k in CLASS_LABELS:
        mask = y_true == k
        if mask.sum() == 0:
            continue
        recalls.append(float(np.mean(y_pred[mask] == k)))

    return float(np.mean(recalls)) if recalls else np.nan

def multiclass_brier_score(y_true, prob):
    y_true = np.asarray(y_true, dtype=int)
    prob = normalize_proba(prob)

    onehot = np.zeros_like(prob)
    for i, y in enumerate(y_true):
        if int(y) in CLASS_LABELS:
            onehot[i, int(y)] = 1.0

    return float(np.mean(np.sum((prob - onehot) ** 2, axis=1)))

def negative_log_likelihood(y_true, prob):
    y_true = np.asarray(y_true, dtype=int)
    prob = normalize_proba(prob)
    clipped = np.clip(prob, 1e-12, 1.0)
    return float(-np.mean(np.log(clipped[np.arange(len(y_true)), y_true])))

def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    prob = normalize_proba(prob)

    pred = np.argmax(prob, axis=1)
    conf = np.max(prob, axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        if mask.sum() == 0:
            continue

        bin_acc = float(correct[mask].mean())
        bin_conf = float(conf[mask].mean())
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)

    return float(ece)

def ordinal_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return float(np.mean(np.abs(y_true - y_pred)))

def simulate_uniform_random_macro_f1(y_true, n_rep=1000, seed=42):
    y_true = np.asarray(y_true, dtype=int)
    rng = np.random.default_rng(seed)

    vals = []
    for _ in range(n_rep):
        y_pred = rng.integers(0, K_CLASSES, size=len(y_true))
        vals.append(macro_f1_score_safe(y_true, y_pred))

    arr = np.asarray(vals, dtype=float)

    return {
        "uniform_random_macro_f1_mean": float(np.mean(arr)),
        "uniform_random_macro_f1_p05": float(np.percentile(arr, 5)),
        "uniform_random_macro_f1_p95": float(np.percentile(arr, 95)),
    }

def probability_skill_metrics(y_true, prob, class_prior_vec, sample_name, horizon, prior_source):
    y_true = np.asarray(y_true, dtype=int)
    prob = normalize_proba(prob)

    n = len(y_true)
    if n == 0:
        raise ValueError("Empty sample passed to probability_skill_metrics.")

    y_pred = np.argmax(prob, axis=1)

    uniform_prob = np.ones((n, K_CLASSES)) / K_CLASSES

    class_prior_vec = np.asarray(class_prior_vec, dtype=float)
    class_prior_vec = np.nan_to_num(class_prior_vec, nan=0.0, posinf=0.0, neginf=0.0)
    class_prior_vec[class_prior_vec < 0] = 0.0
    if class_prior_vec.sum() <= 0:
        class_prior_vec = np.ones(K_CLASSES) / K_CLASSES
    else:
        class_prior_vec = class_prior_vec / class_prior_vec.sum()

    prior_prob = np.tile(class_prior_vec.reshape(1, -1), (n, 1))

    majority_class = int(np.argmax(class_prior_vec))
    majority_pred = np.full(n, majority_class, dtype=int)

    model_brier = multiclass_brier_score(y_true, prob)
    uniform_brier = multiclass_brier_score(y_true, uniform_prob)
    prior_brier = multiclass_brier_score(y_true, prior_prob)

    random_stats = simulate_uniform_random_macro_f1(
        y_true,
        n_rep=RANDOM_BASELINE_REPLICATIONS,
        seed=RANDOM_BASELINE_SEED + int(horizon),
    )

    return {
        "target_horizon": f"{horizon}d",
        "sample": sample_name,
        "n": int(n),
        "model_macro_f1": macro_f1_score_safe(y_true, y_pred),
        "model_balanced_accuracy": balanced_accuracy_score_safe(y_true, y_pred),
        "model_ordinal_mae": ordinal_mae(y_true, y_pred),
        "model_ece": expected_calibration_error(y_true, prob, n_bins=10),
        "model_brier": model_brier,
        "uniform_brier": uniform_brier,
        "class_prior_brier": prior_brier,
        "brier_skill_vs_uniform": float(1.0 - model_brier / uniform_brier) if uniform_brier > 0 else np.nan,
        "brier_skill_vs_class_prior": float(1.0 - model_brier / prior_brier) if prior_brier > 0 else np.nan,
        "model_nll": negative_log_likelihood(y_true, prob),
        "uniform_nll": negative_log_likelihood(y_true, uniform_prob),
        "class_prior_nll": negative_log_likelihood(y_true, prior_prob),
        "majority_class": majority_class,
        "majority_macro_f1": macro_f1_score_safe(y_true, majority_pred),
        "majority_balanced_accuracy": balanced_accuracy_score_safe(y_true, majority_pred),
        "uniform_random_macro_f1_mean": random_stats["uniform_random_macro_f1_mean"],
        "uniform_random_macro_f1_p05": random_stats["uniform_random_macro_f1_p05"],
        "uniform_random_macro_f1_p95": random_stats["uniform_random_macro_f1_p95"],
        "class_prior_source": prior_source,
    }

# ============================================================
# 4. Load probability files and infer true labels
# ============================================================

print("\n" + "=" * 100)
print("Loading probability files and inferring true labels")
print("=" * 100)

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None
target20_col = None
target60_col = None

for _, row in input_index_df.iterrows():
    target_col = str(row["target_col"])
    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col or "20" in target_col:
        p20_path = path
        target20_col = target_col
    elif "60d" in target_col or "60" in target_col:
        p60_path = path
        target60_col = target_col

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files from NOTEBOOK08_INPUT_INDEX.")

p20_raw = read_table_auto_flex(p20_path)
p60_raw = read_table_auto_flex(p60_path)

for name, df in [("20d", p20_raw), ("60d", p60_raw)]:
    missing = [c for c in proba_cols() if c not in df.columns]
    if missing:
        raise ValueError(f"{name} probability file missing probability columns: {missing}")
    if "split" not in df.columns:
        raise ValueError(f"{name} probability file must include split.")
    if "fold_id" not in df.columns:
        raise ValueError(f"{name} probability file must include fold_id.")

print("20d probability path:", p20_path)
print("60d probability path:", p60_path)
print("20d target column:", target20_col)
print("60d target column:", target60_col)

def find_label_column_in_probability_df(prob_df, target_col):
    candidates = [
        target_col,
        f"{target_col}_true",
        f"true_{target_col}",
        f"y_{target_col}",
        "y_true",
        "true_label",
        "target",
        "target_label",
        "label",
        "actual",
        "actual_class",
        "class",
        "regime",
        "regime_class",
        "y",
    ]

    for c in candidates:
        if c in prob_df.columns:
            vals = pd.to_numeric(prob_df[c], errors="coerce")
            valid = vals.dropna()
            if len(valid) > 0 and valid.between(0, 4).mean() > 0.95:
                y = vals.copy()
                y.index = pd.to_datetime(prob_df.index)
                y.index.name = "date"
                return y.sort_index(), f"probability_file_column:{c}"

    return None, None

def search_modeling_dataset_for_target(target_col):
    candidate_files = []

    for root in [MODELING_DIR, DATA_ROOT, PANEL_DIR]:
        root = Path(root)
        if root.exists():
            candidate_files.extend(list(root.rglob("*.csv")))
            candidate_files.extend(list(root.rglob("*.parquet")))

    candidate_files = sorted(
        list(set(candidate_files)),
        key=lambda p: (
            0 if any(s in p.name.lower() for s in ["feature", "label", "model"]) else 1,
            len(p.as_posix()),
        ),
    )

    checked_rows = []

    for p in candidate_files:
        try:
            df = read_table_auto_flex(p)

            if target_col not in df.columns:
                checked_rows.append({
                    "path": str(p),
                    "status": "target_col_not_found",
                    "n_rows": len(df),
                    "has_datetime_index": isinstance(df.index, pd.DatetimeIndex),
                })
                continue

            vals = pd.to_numeric(df[target_col], errors="coerce")
            valid = vals.dropna()

            if len(valid) == 0 or valid.between(0, 4).mean() <= 0.95:
                checked_rows.append({
                    "path": str(p),
                    "status": "target_col_not_valid_regime_label",
                    "n_rows": len(df),
                    "has_datetime_index": isinstance(df.index, pd.DatetimeIndex),
                })
                continue

            if not isinstance(df.index, pd.DatetimeIndex):
                checked_rows.append({
                    "path": str(p),
                    "status": "no_datetime_index",
                    "n_rows": len(df),
                    "has_datetime_index": False,
                })
                continue

            year_series = pd.Series(df.index.year)
            if year_series.between(2020, 2030).mean() < 0.50:
                checked_rows.append({
                    "path": str(p),
                    "status": "datetime_index_not_financial_years",
                    "n_rows": len(df),
                    "has_datetime_index": True,
                    "min_date": str(df.index.min()),
                    "max_date": str(df.index.max()),
                })
                continue

            y = vals.copy()
            y.index = pd.to_datetime(df.index)
            y.index.name = "date"
            y = y[~y.index.duplicated(keep="last")]

            checked_rows.append({
                "path": str(p),
                "status": "selected",
                "n_rows": len(df),
                "has_datetime_index": True,
                "min_date": str(y.index.min()),
                "max_date": str(y.index.max()),
            })

            pd.DataFrame(checked_rows).to_csv(
                DIAG_DIR / f"label_search_checked_{safe_name(target_col)}.csv",
                index=False,
            )

            return y.sort_index(), f"modeling_dataset:{p}"

        except Exception as e:
            checked_rows.append({
                "path": str(p),
                "status": f"read_failed:{repr(e)[:120]}",
                "n_rows": np.nan,
                "has_datetime_index": np.nan,
            })

    pd.DataFrame(checked_rows).to_csv(
        DIAG_DIR / f"label_search_checked_{safe_name(target_col)}.csv",
        index=False,
    )

    return None, None

def infer_horizon_from_target_col(target_col):
    target_col = str(target_col)
    if "20" in target_col:
        return 20
    if "60" in target_col:
        return 60
    raise ValueError(f"Could not infer horizon from target column: {target_col}")

def search_taiex_close_series():
    candidate_files = []

    for root in [PANEL_DIR, MODELING_DIR, DATA_ROOT]:
        root = Path(root)
        if root.exists():
            candidate_files.extend(list(root.rglob("*.csv")))
            candidate_files.extend(list(root.rglob("*.parquet")))

    preferred_tokens = [
        "taiexclose",
        "taiex_close",
        "taiex",
        "twii",
        "taiwancapitalizationweightedstockindex",
    ]

    for p in sorted(list(set(candidate_files)), key=lambda x: len(x.as_posix())):
        try:
            df = read_table_auto_flex(p)
            if not isinstance(df.index, pd.DatetimeIndex):
                continue

            for c in df.columns:
                cn = normalize_name(c)
                if not any(tok in cn for tok in preferred_tokens):
                    continue

                s = pd.to_numeric(df[c], errors="coerce").dropna()
                if len(s) < 100:
                    continue

                if s.median() > 100:
                    out = pd.to_numeric(df[c], errors="coerce")
                    out.index = pd.to_datetime(df.index)
                    out.index.name = "date"
                    out = out[~out.index.duplicated(keep="last")]
                    return out.sort_index(), p, c

        except Exception:
            continue

    return None, None, None

def construct_regime_label_from_taiex_close(close_series, horizon):
    close = pd.Series(close_series).sort_index().astype(float)
    fwd = close.shift(-horizon) / close - 1.0

    y = pd.Series(index=close.index, dtype=float)
    y.loc[fwd < -0.10] = 0
    y.loc[(fwd >= -0.10) & (fwd < -0.03)] = 1
    y.loc[(fwd >= -0.03) & (fwd < 0.03)] = 2
    y.loc[(fwd >= 0.03) & (fwd < 0.10)] = 3
    y.loc[fwd >= 0.10] = 4

    y.index.name = "date"
    return y

def get_true_labels(prob_df, target_col):
    y, source = find_label_column_in_probability_df(prob_df, target_col)
    if y is not None:
        return y, source

    y, source = search_modeling_dataset_for_target(target_col)
    if y is not None:
        return y, source

    horizon = infer_horizon_from_target_col(target_col)
    taiex_close, close_path, close_col = search_taiex_close_series()
    if taiex_close is not None:
        y = construct_regime_label_from_taiex_close(taiex_close, horizon=horizon)
        return y, f"constructed_from_taiex_close:{close_path}:{close_col}"

    raise ValueError(
        f"Unable to find or construct true labels for {target_col}. "
        "Please verify AURORA_TWETF_features_with_labels.csv has a date column and target labels."
    )

y20_all, y20_source = get_true_labels(p20_raw, target20_col)
y60_all, y60_source = get_true_labels(p60_raw, target60_col)

print("20d label source:", y20_source)
print("60d label source:", y60_source)
print("20d label date range:", y20_all.index.min().date(), "to", y20_all.index.max().date())
print("60d label date range:", y60_all.index.min().date(), "to", y60_all.index.max().date())

# Latest-fold pooled test predictions.
p20_test_latest = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test_latest = latest_fold_deduplicate(p60_raw, split_filter=["test"])

strict_dates = (
    p20_test_latest.index
    .intersection(p60_test_latest.index)
    .sort_values()
)

strict_dates = strict_dates[
    (strict_dates >= pd.Timestamp(STRICT_START))
    & (strict_dates <= pd.Timestamp(STRICT_END))
]

if len(strict_dates) == 0:
    raise ValueError("No common strict-test dates found for 20d and 60d probability sources.")

print("Common strict-test dates:", len(strict_dates), strict_dates.min().date(), "to", strict_dates.max().date())

def make_probability_samples(prob_latest, y_all):
    y_all = y_all[~y_all.index.duplicated(keep="last")].copy()

    y = y_all.reindex(prob_latest.index)
    valid = y.notna()

    pooled = prob_latest.loc[valid].copy()
    y_pooled = y.loc[valid].astype(int)

    strict_idx = pooled.index.intersection(strict_dates).sort_values()
    strict = pooled.loc[strict_idx].copy()
    y_strict = y_pooled.loc[strict_idx].astype(int)

    return {
        "pooled_test_folds": (pooled, y_pooled),
        "strict_test_overlap": (strict, y_strict),
    }

samples_20 = make_probability_samples(p20_test_latest, y20_all)
samples_60 = make_probability_samples(p60_test_latest, y60_all)

print("20d pooled/strict n:", len(samples_20["pooled_test_folds"][1]), len(samples_20["strict_test_overlap"][1]))
print("60d pooled/strict n:", len(samples_60["pooled_test_folds"][1]), len(samples_60["strict_test_overlap"][1]))

def class_prior_for_horizon(prob_raw, y_all, sample_index):
    source = None
    prior_index = None

    if "split" in prob_raw.columns:
        val_idx = prob_raw[prob_raw["split"] == "validation"].index
        y_val = y_all.reindex(val_idx).dropna()
        if len(y_val) >= 20:
            prior_index = y_val.index
            source = "validation_split_labels"

    if prior_index is None and "split" in prob_raw.columns:
        oos_idx = prob_raw[prob_raw["split"].isin(["validation", "test"])].index
        oos_idx = pd.DatetimeIndex(oos_idx).sort_values()
        pre_idx = oos_idx[oos_idx < pd.Timestamp(STRICT_START)]
        y_pre = y_all.reindex(pre_idx).dropna()
        if len(y_pre) >= 20:
            prior_index = y_pre.index
            source = "pre_strict_validation_test_labels"

    if prior_index is None:
        prior_index = sample_index
        source = "sample_frequency_fallback"

    y_prior = y_all.reindex(prior_index).dropna().astype(int)
    counts = np.array([(y_prior == k).sum() for k in CLASS_LABELS], dtype=float)

    # Small smoothing avoids zero-probability log-loss issues.
    counts = counts + 1e-8
    prior_vec = counts / counts.sum()

    return prior_vec, source

prob_skill_rows = []

for horizon, prob_raw, samples, y_all, target_col, label_source in [
    (20, p20_raw, samples_20, y20_all, target20_col, y20_source),
    (60, p60_raw, samples_60, y60_all, target60_col, y60_source),
]:
    default_sample_index = samples["pooled_test_folds"][1].index
    class_prior_vec, prior_source = class_prior_for_horizon(
        prob_raw=prob_raw,
        y_all=y_all,
        sample_index=default_sample_index,
    )

    for sample_name, (prob_df, y_sample) in samples.items():
        if len(y_sample) == 0:
            print(f"WARNING: empty probability-skill sample for {horizon}d {sample_name}")
            continue

        prob = normalize_proba(prob_df[proba_cols()].values)

        row = probability_skill_metrics(
            y_true=y_sample.values,
            prob=prob,
            class_prior_vec=class_prior_vec,
            sample_name=sample_name,
            horizon=horizon,
            prior_source=prior_source,
        )

        row["target_col"] = target_col
        row["label_source"] = label_source
        prob_skill_rows.append(row)

s25b = pd.DataFrame(prob_skill_rows)

if s25b.empty:
    raise RuntimeError(
        "Table S25b is empty. Probability dates and label dates still do not overlap. "
        "Check AURORA_TWETF_features_with_labels.csv date parsing."
    )

s25b_cols = [
    "target_horizon",
    "sample",
    "n",
    "model_macro_f1",
    "model_balanced_accuracy",
    "model_ordinal_mae",
    "model_ece",
    "model_brier",
    "uniform_brier",
    "class_prior_brier",
    "brier_skill_vs_uniform",
    "brier_skill_vs_class_prior",
    "model_nll",
    "uniform_nll",
    "class_prior_nll",
    "majority_class",
    "majority_macro_f1",
    "majority_balanced_accuracy",
    "uniform_random_macro_f1_mean",
    "uniform_random_macro_f1_p05",
    "uniform_random_macro_f1_p95",
    "class_prior_source",
    "label_source",
    "target_col",
]
s25b = s25b[[c for c in s25b_cols if c in s25b.columns]]

write_table(s25b, "table_S25b_probability_skill_baselines")
write_rounded_table(s25b, "table_S25b_probability_skill_baselines")

print("\nTable S25b preview:")
print(s25b.round(6).to_string(index=False))

# ============================================================
# 5. Table S41: market-state variable observability and lag convention
# ============================================================

print("\n" + "=" * 100)
print("Creating Table S41: market-state variable observability and lag convention")
print("=" * 100)

s41_rows = [
    {
        "variable_group": "Taiwan ETF prices/returns",
        "variables": "0050, 006208, 00692, 00881",
        "market_or_source_region": "Taiwan",
        "same_calendar_date_observable_before_taiwan_decision": "Yes, if forecast timestamp is after Taiwan close",
        "leakage_safe_convention": "Use latest Taiwan ETF close/return observable before the forecast timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed values; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "Taiwan market index",
        "variables": "TAIEX",
        "market_or_source_region": "Taiwan",
        "same_calendar_date_observable_before_taiwan_decision": "Yes, if forecast timestamp is after Taiwan close",
        "leakage_safe_convention": "Use latest TAIEX close observable before the forecast timestamp; future TAIEX values are used only for label construction during training/evaluation.",
        "holiday_missing_value_handling": "Carry forward only previously observed values; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "U.S. broad and technology markets",
        "variables": "S&P 500, NASDAQ",
        "market_or_source_region": "United States",
        "same_calendar_date_observable_before_taiwan_decision": "No for same-calendar-date Taiwan decisions",
        "leakage_safe_convention": "Lag to latest U.S. close observable before the Taiwan decision timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed U.S. close; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "U.S. semiconductor proxy",
        "variables": "SOXX",
        "market_or_source_region": "United States",
        "same_calendar_date_observable_before_taiwan_decision": "No for same-calendar-date Taiwan decisions",
        "leakage_safe_convention": "Lag to latest U.S. close observable before the Taiwan decision timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed U.S. close; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "U.S. volatility proxy",
        "variables": "VIX",
        "market_or_source_region": "United States",
        "same_calendar_date_observable_before_taiwan_decision": "No for same-calendar-date Taiwan decisions",
        "leakage_safe_convention": "Lag to latest U.S. close observable before the Taiwan decision timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed U.S. close; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "U.S. interest-rate proxy",
        "variables": "U.S. 10-year Treasury yield",
        "market_or_source_region": "United States",
        "same_calendar_date_observable_before_taiwan_decision": "No for same-calendar-date Taiwan decisions",
        "leakage_safe_convention": "Lag to latest U.S. value observable before the Taiwan decision timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed U.S. value; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "Regional Asian equity markets",
        "variables": "Nikkei 225, Hang Seng, KOSPI",
        "market_or_source_region": "Japan, Hong Kong, Korea",
        "same_calendar_date_observable_before_taiwan_decision": "Depends on local close time and decision timestamp",
        "leakage_safe_convention": "Use latest regional-market close observable before the Taiwan decision timestamp; if uncertain, apply one-trading-day lag.",
        "holiday_missing_value_handling": "Carry forward only previously observed regional close; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
    {
        "variable_group": "Foreign-exchange proxy",
        "variables": "USD/TWD",
        "market_or_source_region": "FX / Taiwan data source",
        "same_calendar_date_observable_before_taiwan_decision": "Depends on data timestamp",
        "leakage_safe_convention": "Use latest observable USD/TWD value before the Taiwan decision timestamp.",
        "holiday_missing_value_handling": "Carry forward only previously observed value; do not backfill from the future.",
        "used_directly_as_allocation_rule": "No",
        "verification_status": "Convention table; verify against feature-construction pipeline before final submission.",
    },
]

s41 = pd.DataFrame(s41_rows)

write_table(s41, "table_S41_feature_observability_lag_convention")

print("\nTable S41 preview:")
print(s41.to_string(index=False))

# ============================================================
# 6. Bootstrap block-length sensitivity helpers
# ============================================================

def safe_index_label(x):
    if hasattr(x, "date"):
        return str(x.date())
    return str(x)

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    drawdown = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float((r.mean() / daily_vol) * np.sqrt(ANNUALIZATION_DAYS)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((r.mean() / downside_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": safe_index_label(r.index.min()),
        "end_date": safe_index_label(r.index.max()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
        "final_equity": float(equity.iloc[-1]),
    }

def paired_difference_metrics(strategy_returns, comparator_returns):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()
    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    ms = performance_metrics_from_returns(s)
    mc = performance_metrics_from_returns(c)

    return {
        "n_days": int(len(s)),
        "diff_total_return": ms["total_return"] - mc["total_return"],
        "diff_sharpe": ms["sharpe_ratio"] - mc["sharpe_ratio"],
        "diff_sortino": ms["sortino_ratio"] - mc["sortino_ratio"],
        "drawdown_improvement": ms["max_drawdown"] - mc["max_drawdown"],
        "strategy_total_return": ms["total_return"],
        "comparator_total_return": mc["total_return"],
        "strategy_sharpe": ms["sharpe_ratio"],
        "comparator_sharpe": mc["sharpe_ratio"],
        "strategy_sortino": ms["sortino_ratio"],
        "comparator_sortino": mc["sortino_ratio"],
        "strategy_max_drawdown": ms["max_drawdown"],
        "comparator_max_drawdown": mc["max_drawdown"],
    }

def circular_block_indices(n, block_length, rng):
    idx = []
    while len(idx) < n:
        start = int(rng.integers(0, n))
        idx.extend([(start + j) % n for j in range(block_length)])
    return np.asarray(idx[:n], dtype=int)

def paired_circular_block_bootstrap(strategy_returns, comparator_returns, n_rep, block_length, seed):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()
    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    n = len(s)
    if n <= block_length:
        raise ValueError(f"Too few observations for block bootstrap: n={n}, block_length={block_length}")

    observed = paired_difference_metrics(s, c)

    rng = np.random.default_rng(seed)
    metrics = [
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
    ]

    dist = {m: [] for m in metrics}
    s_values = s.values
    c_values = c.values

    for _ in range(n_rep):
        idx = circular_block_indices(n=n, block_length=block_length, rng=rng)

        bs = pd.Series(s_values[idx]).reset_index(drop=True)
        bc = pd.Series(c_values[idx]).reset_index(drop=True)

        bdiff = paired_difference_metrics(bs, bc)

        for m in metrics:
            dist[m].append(bdiff[m])

    rows = []

    for m in metrics:
        arr = np.asarray(dist[m], dtype=float)
        arr = arr[np.isfinite(arr)]

        ci_low, ci_high = np.percentile(arr, [2.5, 97.5])
        obs = float(observed[m])

        if ci_low > 0:
            result = "Significant positive"
        elif ci_high < 0:
            result = "Significant negative"
        else:
            result = "Not significant"

        rows.append({
            "metric": m,
            "observed_difference": obs,
            "ci95_lower": float(ci_low),
            "ci95_upper": float(ci_high),
            "result": result,
            "bootstrap_replications": int(n_rep),
            "block_length": int(block_length),
            "ci_type": "percentile",
        })

    return pd.DataFrame(rows)

def extract_return_series_from_df(df, label_candidates, return_col_candidates=None):
    if isinstance(label_candidates, str):
        label_candidates = [label_candidates]

    if return_col_candidates is None:
        return_col_candidates = [
            "net_return",
            "daily_return",
            "return",
            "strategy_return",
            "returns",
        ]

    # Long format.
    name_cols = [
        "strategy_control",
        "policy_name",
        "strategy",
        "strategy_name",
        "label",
        "control",
        "portfolio",
    ]

    for name_col in name_cols:
        if name_col not in df.columns:
            continue

        available_return_cols = [c for c in return_col_candidates if c in df.columns]
        if not available_return_cols:
            continue

        ret_col = available_return_cols[0]
        names_norm = df[name_col].astype(str).map(normalize_name)

        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = names_norm == nlab
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                s = s[~s.index.duplicated(keep="last")]
                if len(s) >= 50:
                    return s.sort_index().astype(float)

        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = names_norm.map(lambda x: nlab in x or x in nlab)
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                s = s[~s.index.duplicated(keep="last")]
                if len(s) >= 50:
                    return s.sort_index().astype(float)

    # Wide format.
    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c in df.columns:
            nc = normalize_name(c)
            if nc == nlab or nlab in nc or nc in nlab:
                s = pd.to_numeric(df[c], errors="coerce").dropna()
                if not isinstance(s.index, pd.DatetimeIndex):
                    try:
                        s.index = pd.to_datetime(s.index)
                    except Exception:
                        continue
                s = s[~s.index.duplicated(keep="last")]
                if len(s) >= 50:
                    return s.sort_index().astype(float)

    raise ValueError(f"Could not find return series for labels {label_candidates}")

def search_return_file_for_labels(label_a, label_b, manual_path=None):
    if manual_path is not None:
        p = Path(manual_path)
        if not p.exists():
            raise FileNotFoundError(f"Manual source-aware return matrix path does not exist: {p}")
        df = read_table_auto_flex(p)
        a = extract_return_series_from_df(df, label_a)
        b = extract_return_series_from_df(df, label_b)
        common = a.index.intersection(b.index).sort_values()
        return p, a.loc[common], b.loc[common]

    patterns = [
        "*source_aware*strict*test*return*matrix*.parquet",
        "*source_aware*strict*test*return*matrix*.csv",
        "*source*aware*strict*test*return*.parquet",
        "*source*aware*strict*test*return*.csv",
        "*notebook13B*return*matrix*.parquet",
        "*notebook13B*return*matrix*.csv",
        "*notebook13B*return*.parquet",
        "*notebook13B*return*.csv",
        "*source*aware*return*.parquet",
        "*source*aware*return*.csv",
        "*return*matrix*.parquet",
        "*return*matrix*.csv",
        "*returns*.parquet",
        "*returns*.csv",
    ]

    candidates = []
    for root in [OUTPUT_ROOT, PUBLICATION_ROOT]:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            candidates.extend(list(root.rglob(pat)))

    candidates = list(set([p for p in candidates if p.exists() and p.is_file()]))

    def candidate_priority(p):
        path = p.as_posix().lower()
        return (
            0 if "13b" in path else 1,
            0 if "source" in path and "aware" in path else 1,
            0 if "matrix" in path else 1,
            0 if p.suffix.lower() == ".parquet" else 1,
            len(path),
        )

    candidates = sorted(candidates, key=candidate_priority)

    checked = []

    for p in candidates:
        try:
            df = read_table_auto_flex(p)

            # Must have a plausible date index.
            if not isinstance(df.index, pd.DatetimeIndex):
                checked.append({
                    "path": str(p),
                    "status": "no_datetime_index",
                    "common_n": 0,
                })
                continue

            a = extract_return_series_from_df(df, label_a)
            b = extract_return_series_from_df(df, label_b)
            common = a.index.intersection(b.index).sort_values()

            checked.append({
                "path": str(p),
                "status": "series_found",
                "common_n": len(common),
                "start": str(common.min()) if len(common) else "",
                "end": str(common.max()) if len(common) else "",
            })

            if len(common) >= 100:
                pd.DataFrame(checked).to_csv(
                    DIAG_DIR / "source_aware_return_file_search_checked.csv",
                    index=False,
                )
                return p, a.loc[common], b.loc[common]

        except Exception as e:
            checked.append({
                "path": str(p),
                "status": f"failed:{repr(e)[:160]}",
                "common_n": 0,
            })

    pd.DataFrame(checked).to_csv(
        DIAG_DIR / "source_aware_return_file_search_checked.csv",
        index=False,
    )

    return None, None, None

def find_notebook18_returns(manual_path=None):
    if manual_path is not None:
        p = Path(manual_path)
        if p.exists():
            return p
        raise FileNotFoundError(f"Manual Notebook 18 returns path does not exist: {p}")

    candidates = []
    for root in [
        OUTPUT_ROOT / "aurora_exposure_matched_lambda_reduced_universe",
        OUTPUT_ROOT,
        PUBLICATION_ROOT,
    ]:
        root = Path(root)
        if root.exists():
            candidates.extend(list(root.rglob("all_notebook18_returns.parquet")))
            candidates.extend(list(root.rglob("all_notebook18_returns.csv")))

    candidates = list(set([p for p in candidates if p.exists() and p.is_file()]))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)

    return candidates[0] if candidates else None

# ============================================================
# 7. Generate Table S42
# ============================================================

print("\n" + "=" * 100)
print("Loading return series and generating Table S42")
print("=" * 100)

bootstrap_specs = []

# Main source-aware comparison: AURORA10-UAMV-B vs ROMA-P4.
src_path, aurora_main_returns, roma_p4_returns = search_return_file_for_labels(
    label_a=[
        "AURORA10-UAMV-B",
        "AURORA10_UAMV_B",
        "AURORA10-UAMV-B original",
        "AURORA10_UAMV_B_more60_defensive",
    ],
    label_b=[
        "ROMA-P4",
        "ROMA_P4",
    ],
    manual_path=MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH,
)

if src_path is not None:
    aurora_main_returns = aurora_main_returns.loc[
        (aurora_main_returns.index >= pd.Timestamp(STRICT_START))
        & (aurora_main_returns.index <= pd.Timestamp(STRICT_END))
    ].dropna()

    roma_p4_returns = roma_p4_returns.loc[
        (roma_p4_returns.index >= pd.Timestamp(STRICT_START))
        & (roma_p4_returns.index <= pd.Timestamp(STRICT_END))
    ].dropna()

    common_main = aurora_main_returns.index.intersection(roma_p4_returns.index).sort_values()
    aurora_main_returns = aurora_main_returns.loc[common_main]
    roma_p4_returns = roma_p4_returns.loc[common_main]

    print("Found source-aware return file:", src_path)
    print("AURORA10-UAMV-B vs ROMA-P4 common dates:", len(common_main))

    bootstrap_specs.append({
        "comparison": "AURORA10-UAMV-B vs ROMA-P4",
        "strategy_name": "AURORA10-UAMV-B",
        "comparator_name": "ROMA-P4",
        "strategy_returns": aurora_main_returns,
        "comparator_returns": roma_p4_returns,
        "source_file": str(src_path),
    })
else:
    warning_text = (
        "WARNING: Could not locate a source-aware return file containing both "
        "AURORA10-UAMV-B and ROMA-P4. Table S42 will omit this comparison. "
        "Set MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH and rerun if needed."
    )
    print(warning_text)
    (DIAG_DIR / "missing_source_aware_return_matrix_warning.txt").write_text(
        warning_text,
        encoding="utf-8",
    )

# Notebook 18 attribution comparison:
# Original dynamic AURORA vs exposure-matched constant-lambda AURORA.
notebook18_returns_path = find_notebook18_returns(MANUAL_NOTEBOOK18_RETURNS_PATH)

if notebook18_returns_path is not None:
    notebook18_df = read_table_auto_flex(notebook18_returns_path)

    dynamic_returns = extract_return_series_from_df(
        notebook18_df,
        ["Original dynamic AURORA"],
    )

    constant_lambda_returns = extract_return_series_from_df(
        notebook18_df,
        ["Exposure-matched constant-lambda AURORA"],
    )

    dynamic_returns = dynamic_returns.loc[
        (dynamic_returns.index >= pd.Timestamp(STRICT_START))
        & (dynamic_returns.index <= pd.Timestamp(STRICT_END))
    ].dropna()

    constant_lambda_returns = constant_lambda_returns.loc[
        (constant_lambda_returns.index >= pd.Timestamp(STRICT_START))
        & (constant_lambda_returns.index <= pd.Timestamp(STRICT_END))
    ].dropna()

    common_n18 = dynamic_returns.index.intersection(constant_lambda_returns.index).sort_values()
    dynamic_returns = dynamic_returns.loc[common_n18]
    constant_lambda_returns = constant_lambda_returns.loc[common_n18]

    print("Found Notebook 18 return file:", notebook18_returns_path)
    print("Original dynamic vs constant-lambda common dates:", len(common_n18))

    bootstrap_specs.append({
        "comparison": "Original dynamic AURORA vs exposure-matched constant-lambda AURORA",
        "strategy_name": "Original dynamic AURORA",
        "comparator_name": "Exposure-matched constant-lambda AURORA",
        "strategy_returns": dynamic_returns,
        "comparator_returns": constant_lambda_returns,
        "source_file": str(notebook18_returns_path),
    })
else:
    warning_text = (
        "WARNING: Could not locate Notebook 18 all_notebook18_returns file. "
        "Table S42 will omit the constant-lambda comparison."
    )
    print(warning_text)
    (DIAG_DIR / "missing_notebook18_returns_warning.txt").write_text(
        warning_text,
        encoding="utf-8",
    )

metric_label_map = {
    "diff_total_return": "Total return",
    "diff_sharpe": "Sharpe",
    "diff_sortino": "Sortino",
    "drawdown_improvement": "Max-drawdown improvement",
}

s42_frames = []

if bootstrap_specs:
    for spec_idx, spec in enumerate(bootstrap_specs):
        for block_length in BOOTSTRAP_BLOCK_LENGTHS:
            print("Bootstrap:", spec["comparison"], "| block length:", block_length)

            boot_df = paired_circular_block_bootstrap(
                strategy_returns=spec["strategy_returns"],
                comparator_returns=spec["comparator_returns"],
                n_rep=BOOTSTRAP_REPLICATIONS,
                block_length=block_length,
                seed=BOOTSTRAP_RANDOM_SEED + 1000 * spec_idx + block_length,
            )

            boot_df.insert(0, "comparison", spec["comparison"])
            boot_df.insert(1, "strategy_name", spec["strategy_name"])
            boot_df.insert(2, "comparator_name", spec["comparator_name"])
            boot_df["source_file"] = spec["source_file"]

            s42_frames.append(boot_df)

if s42_frames:
    s42 = pd.concat(s42_frames, ignore_index=True)
    s42["metric_label"] = s42["metric"].map(metric_label_map).fillna(s42["metric"])

    s42_cols = [
        "comparison",
        "metric_label",
        "block_length",
        "observed_difference",
        "ci95_lower",
        "ci95_upper",
        "result",
        "bootstrap_replications",
        "ci_type",
        "strategy_name",
        "comparator_name",
        "metric",
        "source_file",
    ]
    s42 = s42[[c for c in s42_cols if c in s42.columns]]
else:
    s42 = pd.DataFrame(
        columns=[
            "comparison",
            "metric_label",
            "block_length",
            "observed_difference",
            "ci95_lower",
            "ci95_upper",
            "result",
            "bootstrap_replications",
            "ci_type",
            "strategy_name",
            "comparator_name",
            "metric",
            "source_file",
        ]
    )

write_table(s42, "table_S42_bootstrap_block_length_sensitivity")
write_rounded_table(s42, "table_S42_bootstrap_block_length_sensitivity")

# Compact summary table.
summary_rows = []

if not s42.empty:
    for (comparison, metric_label), grp in s42.groupby(["comparison", "metric_label"], sort=False):
        grp = grp.sort_values("block_length")

        supported_blocks = grp.loc[grp["result"] == "Significant positive", "block_length"].astype(int).tolist()
        negative_blocks = grp.loc[grp["result"] == "Significant negative", "block_length"].astype(int).tolist()
        nonsig_blocks = grp.loc[grp["result"] == "Not significant", "block_length"].astype(int).tolist()

        summary_rows.append({
            "comparison": comparison,
            "metric": metric_label,
            "block_length_results": "; ".join(
                f"{int(row['block_length'])}:{row['result']}" for _, row in grp.iterrows()
            ),
            "num_significant_positive_blocks": len(supported_blocks),
            "num_significant_negative_blocks": len(negative_blocks),
            "num_not_significant_blocks": len(nonsig_blocks),
            "significant_positive_block_lengths": ",".join(map(str, supported_blocks)) if supported_blocks else "",
            "significant_negative_block_lengths": ",".join(map(str, negative_blocks)) if negative_blocks else "",
            "not_significant_block_lengths": ",".join(map(str, nonsig_blocks)) if nonsig_blocks else "",
        })

s42b = pd.DataFrame(summary_rows)

if s42b.empty:
    s42b = pd.DataFrame(
        columns=[
            "comparison",
            "metric",
            "block_length_results",
            "num_significant_positive_blocks",
            "num_significant_negative_blocks",
            "num_not_significant_blocks",
            "significant_positive_block_lengths",
            "significant_negative_block_lengths",
            "not_significant_block_lengths",
        ]
    )

write_table(s42b, "table_S42b_bootstrap_block_length_summary")
write_rounded_table(s42b, "table_S42b_bootstrap_block_length_summary")

print("\nTable S42 preview:")
if not s42.empty:
    print(s42.round(6).to_string(index=False))
else:
    print("Table S42 is empty because no required return series were found.")

print("\nTable S42b preview:")
if not s42b.empty:
    print(s42b.to_string(index=False))
else:
    print("Table S42b is empty because no required return series were found.")

# ============================================================
# 8. Interpretation helper
# ============================================================

interpretation_rows = []

# S25b interpretation.
for _, row in s25b.iterrows():
    if row["brier_skill_vs_class_prior"] > 0:
        skill_interp = "positive Brier skill versus class prior"
    elif row["brier_skill_vs_class_prior"] < 0:
        skill_interp = "negative Brier skill versus class prior"
    else:
        skill_interp = "approximately zero Brier skill versus class prior"

    interpretation_rows.append({
        "section": "S25b",
        "item": f"{row['target_horizon']} {row['sample']}",
        "finding": skill_interp,
        "evidence": (
            f"Model Brier={row['model_brier']:.4f}, "
            f"uniform Brier={row['uniform_brier']:.4f}, "
            f"class-prior Brier={row['class_prior_brier']:.4f}, "
            f"Brier skill vs prior={row['brier_skill_vs_class_prior']:.4f}."
        ),
    })

# S42 interpretation.
if not s42b.empty:
    for _, row in s42b.iterrows():
        interpretation_rows.append({
            "section": "S42",
            "item": f"{row['comparison']} | {row['metric']}",
            "finding": row["block_length_results"],
            "evidence": (
                f"Positive blocks={row['significant_positive_block_lengths']}; "
                f"negative blocks={row['significant_negative_block_lengths']}; "
                f"non-significant blocks={row['not_significant_block_lengths']}."
            ),
        })
else:
    interpretation_rows.append({
        "section": "S42",
        "item": "bootstrap block-length sensitivity",
        "finding": "not generated",
        "evidence": (
            "Required return series were not found automatically. "
            "Set MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH and/or MANUAL_NOTEBOOK18_RETURNS_PATH and rerun."
        ),
    })

interpretation_df = pd.DataFrame(interpretation_rows)
write_table(interpretation_df, "notebook19_interpretation_helper")

# ============================================================
# 9. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "19_AURORA_probability_skill_observability_bootstrap_sensitivity_corrected",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Generate supplementary diagnostics for probability-skill baselines, "
        "feature observability convention, and bootstrap block-length sensitivity."
    ),
    "input_paths": {
        "notebook08_or_allocation_input_index": str(NOTEBOOK08_INPUT_INDEX),
        "probability_20d_path": str(p20_path),
        "probability_60d_path": str(p60_path),
        "manual_source_aware_return_matrix_path": str(MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH) if MANUAL_SOURCE_AWARE_RETURN_MATRIX_PATH else None,
        "manual_notebook18_returns_path": str(MANUAL_NOTEBOOK18_RETURNS_PATH) if MANUAL_NOTEBOOK18_RETURNS_PATH else None,
    },
    "label_sources": {
        "20d": y20_source,
        "60d": y60_source,
    },
    "strict_test_period": {
        "start": STRICT_START,
        "end": STRICT_END,
        "common_probability_dates": int(len(strict_dates)),
    },
    "probability_skill": {
        "class_labels": CLASS_LABELS,
        "random_baseline_replications": RANDOM_BASELINE_REPLICATIONS,
        "random_baseline_seed": RANDOM_BASELINE_SEED,
        "table_S25b_rows": int(len(s25b)),
    },
    "feature_observability_note": (
        "Table S41 documents the intended leakage-safe convention. "
        "It does not by itself prove that the feature-construction notebook applied every lag. "
        "To verify implementation, inspect the raw data and feature-construction notebooks."
    ),
    "bootstrap_block_length_sensitivity": {
        "replications": BOOTSTRAP_REPLICATIONS,
        "block_lengths": BOOTSTRAP_BLOCK_LENGTHS,
        "random_seed": BOOTSTRAP_RANDOM_SEED,
        "ci_type": "percentile",
        "bootstrap_type": "paired circular block bootstrap",
        "comparisons_generated": sorted(s42["comparison"].unique().tolist()) if not s42.empty else [],
        "table_S42_rows": int(len(s42)),
    },
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "diagnostics": str(DIAG_DIR),
        "reports": str(REPORT_RUN_DIR),
        "table_S25b": str(TABLE_RUN_DIR / "table_S25b_probability_skill_baselines.csv"),
        "table_S41": str(TABLE_RUN_DIR / "table_S41_feature_observability_lag_convention.csv"),
        "table_S42": str(TABLE_RUN_DIR / "table_S42_bootstrap_block_length_sensitivity.csv"),
        "table_S42b": str(TABLE_RUN_DIR / "table_S42b_bootstrap_block_length_summary.csv"),
        "interpretation_helper": str(TABLE_RUN_DIR / "notebook19_interpretation_helper.csv"),
    },
    "educational_note": (
        "This notebook performs research diagnostics only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK19_validation_report.json"
validation_report_global_path = REPORT_DIR / f"NOTEBOOK19_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "NOTEBOOK19_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"NOTEBOOK19_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 10. Final summary
# ============================================================

print("\n" + "=" * 100)
print("AURORA-TWETF NOTEBOOK 19 COMPLETE")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("Table S25b:", TABLE_RUN_DIR / "table_S25b_probability_skill_baselines.csv")
print("Table S41:", TABLE_RUN_DIR / "table_S41_feature_observability_lag_convention.csv")
print("Table S42:", TABLE_RUN_DIR / "table_S42_bootstrap_block_length_sensitivity.csv")
print("Table S42b:", TABLE_RUN_DIR / "table_S42b_bootstrap_block_length_summary.csv")
print("Interpretation helper:", TABLE_RUN_DIR / "notebook19_interpretation_helper.csv")
print("Validation report:", validation_report_path)
print("Manifest:", manifest_path)
print("=" * 100)

print("\nRounded Table S25b preview:")
print(s25b.round(6).to_string(index=False))

print("\nTable S41 preview:")
print(s41.to_string(index=False))

print("\nRounded Table S42 preview:")
if not s42.empty:
    print(s42.round(6).to_string(index=False))
else:
    print("Table S42 is empty because required return series were not found.")

print("\nTable S42b preview:")
if not s42b.empty:
    print(s42b.to_string(index=False))
else:
    print("Table S42b is empty because required return series were not found.")

print("\nInterpretation helper preview:")
print(interpretation_df.to_string(index=False))

Mounted at /content/drive
AURORA-TWETF Notebook 19
Probability skill baselines, feature observability, and bootstrap block-length sensitivity
RUN_ID: 20260718_101202
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/probability_skill_observability_bootstrap_sensitivity/run_20260718_101202
NOTEBOOK08_INPUT_INDEX: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv

Loading probability files and inferring true labels
20d probability path: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_20d_E1_validation_weighted_probability_ensemble.parquet
60d probability path: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_60d_E1_validation_weighted_probability_en